# Table 2: Proportion of validated responses
Compute source and WWW NSEC/NSEC3 validation percentages.

`validated percentage = 100 × count(ad_flag == 1) / total domain count`

It uses `source_nsec`, `source_nsec3`, `www_nsec`, and `www_nsec3` from `script/res/` as denominators. The source totals include auxiliary scanning (OpenIntel sometimes have missing data so we conduct extra scanning).

The paper's source percentages was used **OpenINTEL-only totals before auxiliary scanning** (for consistency purpose in case user do not have extra scanning). These user-supplied reference counts are saved in `notebook/reference/20260406_openintel_pre_auxiliary_domcnt.csv`. This notebook displays both the corrected calculation and the historical calculation that reproduces the paper, then lists the required changes to Table 2.



In [8]:
from pathlib import Path
import pandas as pd
from IPython.display import display

DATE = "20260406"
REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "script/res" / f"{DATE}_domcnt_stats.csv").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the dnsnsec26 repository or set REPO_ROOT explicitly.")
RES_DIR = REPO_ROOT / "script/res"
OUTPUT_DIR = REPO_ROOT / "notebook/results"
ZONES = ["com", "net", "org", "umbrella", "tranco", "radar", "majestic"]
LABELS = {"com": ".com", "net": ".net", "org": ".org",
          "umbrella": "Umbrella", "tranco": "Tranco",
          "radar": "Radar", "majestic": "Majestic"}

## Load numerators and denominators
The four inputs are `20260406_adflag_stats.csv`, `20260406_www_adflag_stats.csv`, `20260406_domcnt_stats.csv`, and `20260406_www_domcnt_stats.csv`. Joins use dataset names and record types rather than CSV row positions.

The additional pre-auxiliary reference CSV was supplied by the user, with rows in the same seven-dataset order. See `notebook/reference/README.md` for provenance and interpretation.

In [9]:
source_domains = pd.read_csv(RES_DIR / f"{DATE}_domcnt_stats.csv")
www_domains = pd.read_csv(RES_DIR / f"{DATE}_www_domcnt_stats.csv")
source_adflags = pd.read_csv(RES_DIR / f"{DATE}_adflag_stats.csv")
www_adflags = pd.read_csv(RES_DIR / f"{DATE}_www_adflag_stats.csv")


def domain_totals(frame, prefix):
    if frame["zone"].duplicated().any() or set(frame["zone"]) != set(ZONES):
        raise ValueError(f"{prefix}: expected one domain-count row for each dataset.")
    return frame.set_index("zone").loc[ZONES, ["nsec", "nsec3"]].add_prefix(prefix + "_")


def validated_counts(frame, prefix):
    keys = ["Zone", "Type"]
    if frame.duplicated(keys).any():
        raise ValueError(f"{prefix}: duplicate AD-flag dataset/type rows.")
    expected = {(zone, mechanism) for zone in ZONES for mechanism in ("NSEC", "NSEC3")}
    if set(map(tuple, frame[keys].to_numpy())) != expected:
        raise ValueError(f"{prefix}: expected NSEC and NSEC3 AD-flag rows for every dataset.")
    # CSV column '1' is the count of domains with ad_flag == 1.
    counts = frame.pivot(index="Zone", columns="Type", values="1").loc[ZONES]
    return counts.rename(columns={
        "NSEC": f"{prefix}_nsec_ad1", "NSEC3": f"{prefix}_nsec3_ad1"
    }).rename_axis(index="zone", columns=None)


stats = domain_totals(source_domains, "source").join(domain_totals(www_domains, "www"))
stats = stats.join(validated_counts(source_adflags, "source"))
stats = stats.join(validated_counts(www_adflags, "www"))
values = stats.apply(pd.to_numeric, errors="raise")
if values.isna().any().any() or (values < 0).any().any() or (values % 1 != 0).any().any():
    raise ValueError("Summary counts must be nonnegative integers with no missing values.")
stats = values.astype("int64")

## Corrected Table 2: merged source denominators
This retains the previous calculation. Source AD=1 counts and source domain totals refer to the merged source dataset; WWW uses its corresponding domain totals. Percentages remain numeric on the 0–100 scale, with two-decimal formatting only for display.

In [10]:
for population in ("source", "www"):
    for mechanism in ("nsec", "nsec3"):
        key = f"{population}_{mechanism}"
        numerator = stats[f"{key}_ad1"]
        denominator = stats[key]
        if denominator.le(0).any() or numerator.gt(denominator).any():
            raise ValueError(f"Invalid numerator or domain denominator for {key}.")
        stats[f"{key}_pct"] = 100 * numerator / denominator

# Match the paper's column order: NSEC Source/WWW, then NSEC3 Source/WWW.
percentage_columns = ["source_nsec_pct", "www_nsec_pct", "source_nsec3_pct", "www_nsec3_pct"]
table2 = stats[percentage_columns].copy()
table2.columns = pd.MultiIndex.from_tuples([
    ("NSEC", "Source"), ("NSEC", "WWW"),
    ("NSEC3", "Source"), ("NSEC3", "WWW"),
])
table2.index = table2.index.map(LABELS)
table2.index.name = "Dataset"
formatted_table2 = table2.apply(lambda column: column.map(lambda value: f"{value:.2f}%"))
display(formatted_table2)

NSEC           NSEC3        
          Source     WWW  Source     WWW
Dataset                                 
.com      76.54%  78.16%  81.07%  80.90%
.net      80.72%  81.32%  84.55%  84.50%
.org      81.36%  81.98%  81.63%  81.73%
Umbrella  68.61%  68.57%  74.56%  75.45%
Tranco    62.79%  63.16%  75.72%  77.52%
Radar     62.82%  63.50%  78.50%  80.38%
Majestic  65.05%  66.14%  78.65%  80.46%

## Historical calculation: reproduce the paper's source percentages
Use the same saved source AD=1 numerators with the pre-auxiliary OpenINTEL domain totals. All 14 source percentages then match the paper to two decimals.

This is a reconstruction of the paper's denominator error, **not an independently computed OpenINTEL-only validation rate**: the saved source AD=1 numerators are from the merged dataset. A true OpenINTEL-only rate would require matching OpenINTEL-only AD=1 counts, which are not provided here. The WWW calculation is identical in both tables.

In [11]:
reference_path = REPO_ROOT / "notebook/reference" / f"{DATE}_openintel_pre_auxiliary_domcnt.csv"
reference_domains = pd.read_csv(reference_path)
reference_totals = domain_totals(reference_domains, "source")
reference_totals = reference_totals.apply(pd.to_numeric, errors="raise")
if (reference_totals.isna().any().any() or reference_totals.le(0).any().any()
        or (reference_totals % 1 != 0).any().any()):
    raise ValueError("Reference denominators must be positive integers.")
reference_totals = reference_totals.astype("int64")

historical_stats = stats.copy()
for mechanism in ("nsec", "nsec3"):
    key = f"source_{mechanism}"
    historical_stats[key] = reference_totals[key]
    if historical_stats[f"{key}_ad1"].gt(historical_stats[key]).any():
        raise ValueError(f"AD=1 numerator exceeds reference total for {key}.")
    historical_stats[f"{key}_pct"] = 100 * historical_stats[f"{key}_ad1"] / historical_stats[key]

historical_table2 = historical_stats[percentage_columns].copy()
historical_table2.columns = table2.columns
historical_table2.index = table2.index
formatted_historical_table2 = historical_table2.apply(
    lambda column: column.map(lambda value: f"{value:.2f}%")
)
display(formatted_historical_table2)

paper_source_percentages = pd.DataFrame({
    "source_nsec_pct": [76.57, 80.74, 81.38, 69.40, 62.90, 62.88, 65.12],
    "source_nsec3_pct": [90.15, 90.44, 91.11, 76.80, 80.03, 82.03, 82.52],
}, index=pd.Index(ZONES, name="zone"))
pd.testing.assert_frame_equal(
    historical_stats[paper_source_percentages.columns].round(2),
    paper_source_percentages, check_exact=True,
)
print("All 14 historical source percentages reproduce the paper to two decimals.")

NSEC           NSEC3        
          Source     WWW  Source     WWW
Dataset                                 
.com      76.57%  78.16%  90.15%  80.90%
.net      80.74%  81.32%  90.44%  84.50%
.org      81.38%  81.98%  91.11%  81.73%
Umbrella  69.40%  68.57%  76.80%  75.45%
Tranco    62.90%  63.16%  80.03%  77.52%
Radar     62.88%  63.50%  82.03%  80.38%
Majestic  65.12%  66.14%  82.52%  80.46%

All 14 historical source percentages reproduce the paper to two decimals.


## Required paper correction
Table 2's **14 source cells should be replaced** with the corrected merged-source percentages below. The previous domain-count data and notebook formula are correct; the error was using pre-auxiliary denominators in the paper. The WWW cells remain the same.

The Section 5.2 statement that approximately **77–91% of NSEC3 responses are validated** should also be revised: across source and WWW datasets, the corrected range is approximately **75–85%** (74.56–84.55% before rounding to whole percentages). The discussion of higher authentication rates and the approximate 15% average gap between NSEC3 and NSEC should be rechecked using the corrected values.

The comparison below shows published percentages, replacement percentages, and changes in percentage points (`corrected − paper`, using the displayed two-decimal values).

In [12]:
correction = pd.DataFrame(index=stats.index)
for mechanism, label in (("nsec", "NSEC"), ("nsec3", "NSEC3")):
    key = f"source_{mechanism}_pct"
    correction[f"{label}: paper %"] = paper_source_percentages[key]
    correction[f"{label}: corrected %"] = stats[key].round(2)
    correction[f"{label}: change (pp)"] = (
        correction[f"{label}: corrected %"] - correction[f"{label}: paper %"]
    ).round(2)
correction.index = correction.index.map(LABELS)
correction.index.name = "Dataset"
display(correction.apply(lambda column: column.map(lambda value: f"{value:.2f}")))

all_nsec3_rates = stats[["source_nsec3_pct", "www_nsec3_pct"]]
print(f"Corrected NSEC3 validation range: {all_nsec3_rates.min().min():.2f}%–{all_nsec3_rates.max().max():.2f}%")

,NSEC: paper %,NSEC: corrected %,NSEC: change (pp),NSEC3: paper %,NSEC3: corrected %,NSEC3: change (pp)
Dataset,,,,,,
.com,76.57,76.54,-0.03,90.15,81.07,-9.08
.net,80.74,80.72,-0.02,90.44,84.55,-5.89
.org,81.38,81.36,-0.02,91.11,81.63,-9.48
Umbrella,69.40,68.61,-0.79,76.80,74.56,-2.24
Tranco,62.90,62.79,-0.11,80.03,75.72,-4.31
Radar,62.88,62.82,-0.06,82.03,78.50,-3.53
Majestic,65.12,65.05,-0.07,82.52,78.65,-3.87


Corrected NSEC3 validation range: 74.56%–84.55%


## Export both calculations and the correction list
- `table2_stats.csv` and `table2_formatted.csv`: corrected merged-source calculation, retaining the previous filenames.
- `table2_historical_stats.csv` and `table2_historical_formatted.csv`: historical pre-auxiliary-denominator calculation that reproduces the paper.
- `table2_source_corrections.csv`: published source percentages and their replacements.

Numeric exports retain unrounded calculated percentages; presentation tables use two decimals. Rerunning this cell replaces these exports.

In [14]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
stats.to_csv(OUTPUT_DIR / "table2_stats.csv", index_label="zone")
historical_stats.to_csv(OUTPUT_DIR / "table2_historical_stats.csv", index_label="zone")
print(f"Saved corrected results, historical results, and source corrections to {OUTPUT_DIR}")

Saved corrected results, historical results, and source corrections to /u/yz6me/dnsnsec26/notebook/results
